In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from pathlib import  Path

In [23]:
YEAR = 2019
MONTHS = [6, 7, 8, 9]

files = []
for m in MONTHS:
    pattern = (
        f"/home/users/mendrika/Object-Based-LSTMConv/outputs/initiation/training/"
        f"ci_core_assoc/ci_core_assoc_{YEAR}{m:02d}_t*min.csv"
    )
    files.extend(glob(pattern))

files = sorted(files)

dfs = [pd.read_csv(f, parse_dates=["valid_time"]) for f in files]
df = pd.concat(dfs, ignore_index=True)

print("Total CI samples:", len(df))
print("Lead times:", sorted(df["lead_time_min"].unique()))

Total CI samples: 182972
Lead times: [np.int64(30), np.int64(60), np.int64(90), np.int64(120)]


In [24]:
df

,valid_time,lead_time_min,month,hour,dmin_t0_km,dmin_t_km,excl_010km,excl_015km,excl_020km,excl_025km,...,excl_300km,excl_350km,incl_010km,incl_015km,incl_020km,incl_025km,incl_030km,incl_040km,incl_050km,incl_060km
0,2019-06-01 10:00:00+00:00,30,6,10,205.437469,NaN,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
1,2019-06-01 10:00:00+00:00,30,6,10,405.643921,NaN,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2019-06-01 10:00:00+00:00,30,6,10,460.389832,NaN,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2019-06-01 10:00:00+00:00,30,6,10,76.786674,NaN,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
4,2019-06-01 10:00:00+00:00,30,6,10,NaN,NaN,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182967,2019-09-30 17:30:00+00:00,120,9,17,275.160309,548.947083,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
182968,2019-09-30 17:45:00+00:00,120,9,17,55.155647,0.000000,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
182969,2019-09-30 17:45:00+00:00,120,9,17,207.259827,260.364349,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
182970,2019-09-30 17:45:00+00:00,120,9,17,173.761887,143.610657,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0


# Unconditional inclusion probability (baseline)

Fraction of CI events followed by a core within 25 km.

In [25]:
df.groupby("lead_time_min")["incl_025km"].mean()

lead_time_min
30     0.054916
60     0.122073
90     0.149247
120    0.142951
Name: incl_025km, dtype: float64

# Conditional inclusion

Given no nearby core at t0, what is the probability of a nearby core appearing later

In [29]:
GUARD_BY_LEAD = {
    30: 100,
    60: 180,
    90: 250,
    120: 350,
}

for lead in sorted(df["lead_time_min"].unique()):

    df_lead = df[df["lead_time_min"] == lead]

    # local exclusion (fixed 30 km)
    local_clean = df_lead["excl_030km"] == 0

    # lead-dependent guard exclusion
    guard_km = GUARD_BY_LEAD[lead]
    guard_clean = df_lead[f"excl_{guard_km:03d}km"] == 0

    mask = local_clean & guard_clean

    p_25 = df_lead.loc[mask, "incl_025km"].mean()
    p_40 = df_lead.loc[mask, "incl_040km"].mean()

    n_total = len(df_lead)
    n_clean = mask.sum()

    print(
        f"Lead {lead} min | Guard {guard_km} km | "
        f"N_clean={n_clean}/{n_total} | "
        f"P(incl_25km)={p_25:.4f} | "
        f"P(incl_40km)={p_40:.4f}"
    )


Lead 30 min | Guard 100 km | N_clean=40222/45743 | P(incl_25km)=0.0342 | P(incl_40km)=0.0429
Lead 60 min | Guard 180 km | N_clean=33229/45743 | P(incl_25km)=0.0703 | P(incl_40km)=0.0914
Lead 90 min | Guard 250 km | N_clean=28227/45743 | P(incl_25km)=0.0810 | P(incl_40km)=0.1177
Lead 120 min | Guard 350 km | N_clean=22550/45743 | P(incl_25km)=0.0708 | P(incl_40km)=0.1158
